# Module 093: Asynchronous Python: asyncio

Phase 10: Concurrency & Internals | Duration: 2.5 hours

## Async/await Basics

In [ ]:
import asyncio

async def greet(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"Hello, {name}!"

async def main() -> None:
    result: str = await greet("Alice", 0.5)
    print(result)

asyncio.run(main())

## Creating and Gathering Tasks

In [ ]:
import asyncio
import time
from typing import List

async def fetch_data(id: int, delay: float) -> dict:
    await asyncio.sleep(delay)
    return {"id": id, "data": f"result-{id}"}

async def main() -> None:
    start: float = time.time()

    tasks: List[asyncio.Task] = [
        asyncio.create_task(fetch_data(i, 0.5 * i))
        for i in range(1, 6)
    ]

    results: List[dict] = await asyncio.gather(*tasks)
    elapsed: float = time.time() - start

    for r in results:
        print(r)
    print(f"Total time: {elapsed:.2f}s")

asyncio.run(main())

## Sequential vs Async Comparison

In [ ]:
import asyncio
import time
from typing import List

async def simulate_io(n: int) -> int:
    await asyncio.sleep(0.5)
    return n

def sync_io(n: int) -> int:
    time.sleep(0.5)
    return n

# Sequential
start: float = time.time()
for i in range(5):
    sync_io(i)
print(f"Sequential: {time.time() - start:.2f}s")

# Async
async def run_async() -> None:
    start = time.time()
    tasks = [asyncio.create_task(simulate_io(i)) for i in range(5)]
    await asyncio.gather(*tasks)
    print(f"Async: {time.time() - start:.2f}s")

asyncio.run(run_async())

## Async HTTP with aiohttp

In [ ]:
import asyncio
import aiohttp
from typing import List

async def fetch_status(session: aiohttp.ClientSession, url: str) -> int:
    async with session.get(url) as response:
        return response.status

async def main() -> None:
    urls: List[str] = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/delay/3",
    ]
    async with aiohttp.ClientSession() as session:
        tasks = [asyncio.create_task(fetch_status(session, url)) for url in urls]
        results: List[int] = await asyncio.gather(*tasks)
        print(f"Status codes: {results}")

asyncio.run(main())

## Event Loop Diagram

```
Event Loop Execution Flow:

┌─────────────────────────────────────────────────────────┐
│  Event Loop                                              │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐              │
│  │ Coro A   │  │ Coro B   │  │ Coro C   │              │
│  │ await io │─►│ await io │─►│ await io │─► ...        │
│  └────┬─────┘  └────┬─────┘  └────┬─────┘              │
│       │              │              │                    │
│       ▼              ▼              ▼                    │
│  ┌─────────────────────────────────────┐                │
│  │  I/O Operations (non-blocking)      │                │
│  │  - Network requests                 │                │
│  │  - File I/O                         │                │
│  │  - Timers (sleep)                   │                │
│  └─────────────────────────────────────┘                │
└─────────────────────────────────────────────────────────┘

The event loop runs one coroutine at a time, but switches
between them when they await I/O operations.
```

## When Async Helps

- **I/O-bound tasks**: Web requests, file operations, database queries
- **Many concurrent connections**: Hundreds or thousands of simultaneous operations
- **Not for CPU-bound tasks**: CPU-heavy work blocks the event loop

For CPU-bound work, use multiprocessing instead.